# PRISM Rebuttal - OOD transfer under an MLP probe

The in-distribution MLP run showed that some findings are probe-dependent.
This notebook asks the same question of the paper's headline result:

> On MHIST to PCam, additional source-domain labels **worsen** target
> calibration, with ECE rising from about 0.23 at 1% labels to between 0.44
> and 0.49 at 100% for UNI, VIRCHOW2, GigaPath and H-Optimus-0.

If that trend survives an MLP probe, it is a property of the representations
and the transfer setting. If it does not, the claim belongs to the
linear-probe protocol and the manuscript must say so.

**Setup.** Same four pairs, same stratified subsets, same seeds, same ECE
definition. Temperature fitted on a held-out **source** validation split, as
in the corrected linear run, never on target labels. The only change is the
probe: one hidden layer of width 256 in place of logistic regression.

576 fits. Roughly 40 minutes on GPU. Checkpointed per (model, pair).

In [1]:
import os, gc, glob, time, warnings
import numpy as np, pandas as pd, torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss
from scipy.optimize import minimize_scalar
from google.colab import drive

warnings.filterwarnings('ignore')
drive.mount('/content/drive')

BASE    = '/content/drive/MyDrive/PRISM'
EMB_DIR = f'{BASE}/embeddings'
OUT_DIR = f'{BASE}/results_v2'
CKPT    = f'{OUT_DIR}/ood_mlp_parts'
os.makedirs(CKPT, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

MODELS = ['CLIP','PLIP','CONCH','VIRCHOW2','UNI','GigaPath','H-Optimus-0','MIDNIGHT']
MKEYS  = ['clip','plip','conch','virchow2','uni','gigapath','h_optimus_0','midnight']
M2K    = dict(zip(MODELS, MKEYS))

D2K = {'PCam':'pcam', 'MHIST':'mhist', 'CRC':'crc', 'BRACS':'bracs'}
OOD_PAIRS = [('MHIST','PCam'), ('PCam','MHIST'), ('CRC','BRACS'), ('BRACS','CRC')]

FRACTIONS = [0.01, 0.05, 0.10, 0.25, 0.50, 1.00]
SEEDS     = [42, 123, 456]
N_BINS    = 15

HIDDEN, LR, WEIGHT_DECAY = 256, 1e-3, 1e-4
MAX_EPOCHS, PATIENCE, BATCH = 200, 15, 512

# sorted folder order:
# CRC   ADI=0 BACK=1 DEB=2 LYM=3 MUC=4 MUS=5 NORM=6 STR=7 TUM=8
# BRACS ADH=0 DCIS=1 FEA=2 IC=3 N=4 PB=5 UDH=6
CRC_BIN   = lambda y: (y == 8).astype(int)
BRACS_BIN = lambda y: np.isin(y, [1, 3]).astype(int)

print(torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU only')

Mounted at /content/drive
NVIDIA A100-SXM4-80GB


## 1. Metrics and probe, identical to the other runs

In [2]:
def _ece_edges(conf, correct, edges):
    ece, n = 0.0, len(conf)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf >= lo) & (conf < hi)
        if m.sum() > 0:
            ece += m.sum() * abs(correct[m].mean() - conf[m].mean())
    return float(ece / n)

def conf_correct(proba, y):
    if proba.shape[1] == 2:
        return proba[:, 1], (y == 1).astype(float)
    return proba.max(1), (proba.argmax(1) == y).astype(float)

def ece_fixed(proba, y, n_bins=N_BINS):
    c, k = conf_correct(proba, y)
    return _ece_edges(c, k, np.linspace(0, 1, n_bins + 1))

def ece_adaptive(proba, y, n_bins=N_BINS):
    c, k = conf_correct(proba, y)
    e = np.quantile(c, np.linspace(0, 1, n_bins + 1))
    e[0], e[-1] = 0.0, 1.0 + 1e-9
    e = np.unique(e)
    return ece_fixed(proba, y, n_bins) if len(e) < 3 else _ece_edges(c, k, e)

def softmax_np(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)

def fit_temperature(val_logits, val_y, bounds=(0.1, 10.0)):
    idx = np.arange(len(val_y))
    def nll(T):
        p = softmax_np(val_logits / T)
        return float(-np.log(p[idx, val_y] + 1e-12).mean())
    return float(minimize_scalar(nll, bounds=bounds, method='bounded').x)

def stratified_sample(labels, fraction, seed):
    np.random.seed(seed)
    idx_all = np.arange(len(labels))
    picked = []
    for c in np.unique(labels):
        ci = idx_all[labels == c]
        picked.extend(np.random.choice(ci, size=max(1, int(len(ci) * fraction)),
                                       replace=False))
    return np.array(sorted(picked))

def degeneracy(pred, k):
    cnt = np.bincount(pred, minlength=k)
    return float(cnt.max() / cnt.sum())

def load_emb(mkey, dkey, split):
    p = f'{EMB_DIR}/{mkey}/{dkey}'
    return (np.load(f'{p}/{split}_features.npy', mmap_mode='r'),
            np.load(f'{p}/{split}_labels.npy').astype(int))


class MLPProbe(nn.Module):
    def __init__(self, d_in, n_classes, hidden=HIDDEN):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, hidden), nn.ReLU(),
                                 nn.Linear(hidden, n_classes))
    def forward(self, x):
        return self.net(x)


@torch.no_grad()
def logits_of(model, X, batch=8192):
    model.eval()
    return np.vstack([
        model(torch.as_tensor(np.asarray(X[i:i+batch], dtype=np.float32),
                              device=DEVICE)).float().cpu().numpy()
        for i in range(0, len(X), batch)])


def train_mlp(Xtr, ytr, Xva, yva, n_classes, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    Xt = torch.as_tensor(np.asarray(Xtr, dtype=np.float32), device=DEVICE)
    yt = torch.as_tensor(ytr, dtype=torch.long, device=DEVICE)
    model = MLPProbe(Xt.shape[1], n_classes).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    lossf = nn.CrossEntropyLoss()

    has_val = Xva is not None and yva is not None and len(yva) > 0
    if has_val:
        Xv = torch.as_tensor(np.asarray(Xva, dtype=np.float32), device=DEVICE)
        yv = torch.as_tensor(yva, dtype=torch.long, device=DEVICE)

    n, bs = len(yt), min(BATCH, len(yt))
    best, best_state, bad = float('inf'), None, 0
    for _ in range(MAX_EPOCHS):
        model.train()
        perm = torch.randperm(n, device=DEVICE)
        for i in range(0, n, bs):
            j = perm[i:i+bs]
            loss = lossf(model(Xt[j]), yt[j])
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        if has_val:
            model.eval()
            with torch.no_grad():
                vl = float(lossf(model(Xv), yv))
            if vl < best - 1e-5:
                best, bad = vl, 0
                best_state = {k: v.detach().clone()
                              for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= PATIENCE:
                    break
    if best_state is not None:
        model.load_state_dict(best_state)

    del Xt, yt
    if has_val:
        del Xv, yv
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    return model

print('ready')

ready


## 2. Run the four transfer pairs

In [3]:
def load_side(mk, name):
    dk = D2K[name]
    Xtr, ytr = load_emb(mk, dk, 'train')
    Xte, yte = load_emb(mk, dk, 'test')
    try:
        Xva, yva = load_emb(mk, dk, 'val')
    except FileNotFoundError:
        Xva, yva = None, None
    if name == 'CRC':
        ytr, yte = CRC_BIN(ytr), CRC_BIN(yte)
        yva = None if yva is None else CRC_BIN(yva)
    elif name == 'BRACS':
        ytr, yte = BRACS_BIN(ytr), BRACS_BIN(yte)
        yva = None if yva is None else BRACS_BIN(yva)
    return Xtr, ytr, Xva, yva, Xte, yte


def run_pair(model_name, src, tgt):
    mk = M2K[model_name]
    Xs_tr, ys_tr, Xs_va, ys_va, _, _ = load_side(mk, src)
    _, _, _, _, Xt_te, yt_te = load_side(mk, tgt)
    rows = []

    for frac in FRACTIONS:
        for seed in SEEDS:
            idx = stratified_sample(ys_tr, frac, seed)
            net = train_mlp(np.asarray(Xs_tr[idx]), ys_tr[idx],
                            Xs_va, ys_va, 2, seed)

            Lt    = logits_of(net, Xt_te)
            proba = softmax_np(Lt)
            pred  = proba.argmax(1)

            try:
                auroc = roc_auc_score(yt_te, proba[:, 1])
            except Exception:
                auroc = np.nan

            if Xs_va is not None:
                T  = fit_temperature(logits_of(net, Xs_va), ys_va)
                sp = softmax_np(Lt / T)
                ece_s_fix, ece_s_ada = ece_fixed(sp, yt_te), ece_adaptive(sp, yt_te)
            else:
                T = ece_s_fix = ece_s_ada = np.nan

            rows.append(dict(
                probe='mlp', model=model_name, src=src, tgt=tgt,
                pair=f'{src}->{tgt}', fraction=frac, seed=seed,
                n_train=len(idx), auroc=auroc,
                f1_macro=f1_score(yt_te, pred, average='macro', zero_division=0),
                brier=brier_score_loss(yt_te, proba[:, 1]),
                ece_fixed=ece_fixed(proba, yt_te),
                ece_adaptive=ece_adaptive(proba, yt_te),
                temperature_src=T,
                ece_scaled_fixed=ece_s_fix,
                ece_scaled_adaptive=ece_s_ada,
                degeneracy_share=degeneracy(pred, 2)))

            del net, Lt, proba
            gc.collect()
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()

    del Xs_tr, Xs_va, Xt_te
    gc.collect()
    df = pd.DataFrame(rows)
    df['degenerate'] = df['degeneracy_share'] > 0.99
    return df


t0 = time.time()
for src, tgt in OOD_PAIRS:
    for model_name in MODELS:
        out = f'{CKPT}/{M2K[model_name]}__{D2K[src]}_to_{D2K[tgt]}.csv'
        if os.path.exists(out):
            print(f'  skip: {model_name} {src}->{tgt}')
            continue
        try:
            df = run_pair(model_name, src, tgt)
            df.to_csv(out, index=False)
            s = df.groupby('fraction')['ece_fixed'].mean()
            arrow = 'ARTIYOR' if s.loc[1.00] > s.loc[0.01] else 'azaliyor'
            print(f'{model_name:>12} {src:>6}->{tgt:<6} '
                  f'ECE {s.loc[0.01]:.3f} -> {s.loc[1.00]:.3f}  {arrow}  '
                  f'({time.time()-t0:.0f}s)')
        except Exception as e:
            print(f'{model_name:>12} {src:>6}->{tgt:<6} FAILED: '
                  f'{type(e).__name__}: {e}')
        gc.collect()

parts = sorted(glob.glob(f'{CKPT}/*.csv'))
df_ood_mlp = pd.concat([pd.read_csv(p) for p in parts], ignore_index=True)
df_ood_mlp.to_csv(f'{OUT_DIR}/ood_mlp.csv', index=False)
print(f'\n{len(parts)}/32 pairs, {len(df_ood_mlp)} runs -> ood_mlp.csv')

        CLIP  MHIST->PCam   ECE 0.243 -> 0.394  ARTIYOR  (36s)
        PLIP  MHIST->PCam   ECE 0.192 -> 0.245  ARTIYOR  (62s)
       CONCH  MHIST->PCam   ECE 0.143 -> 0.393  ARTIYOR  (91s)
    VIRCHOW2  MHIST->PCam   ECE 0.091 -> 0.363  ARTIYOR  (124s)
         UNI  MHIST->PCam   ECE 0.048 -> 0.164  ARTIYOR  (149s)
    GigaPath  MHIST->PCam   ECE 0.095 -> 0.187  ARTIYOR  (180s)
 H-Optimus-0  MHIST->PCam   ECE 0.099 -> 0.480  ARTIYOR  (209s)
    MIDNIGHT  MHIST->PCam   ECE 0.117 -> 0.109  azaliyor  (237s)
        CLIP   PCam->MHIST  ECE 0.332 -> 0.283  azaliyor  (490s)
        PLIP   PCam->MHIST  ECE 0.536 -> 0.284  azaliyor  (640s)
       CONCH   PCam->MHIST  ECE 0.206 -> 0.255  ARTIYOR  (770s)
    VIRCHOW2   PCam->MHIST  ECE 0.171 -> 0.163  azaliyor  (964s)
         UNI   PCam->MHIST  ECE 0.274 -> 0.279  ARTIYOR  (1068s)
    GigaPath   PCam->MHIST  ECE 0.160 -> 0.217  ARTIYOR  (1186s)
 H-Optimus-0   PCam->MHIST  ECE 0.328 -> 0.299  azaliyor  (1306s)
    MIDNIGHT   PCam->MHIST  ECE 0.3

## 3. Does reverse OOD scaling survive?

The claim covers UNI, VIRCHOW2, GigaPath and H-Optimus-0 on MHIST to PCam.
Monotone increase in raw ECE with source label fraction is what has to hold.

In [4]:
lin = pd.read_csv(f'{OUT_DIR}/ood_all_v2.csv')
lin['probe'] = 'linear'
mlp = df_ood_mlp

CLAIM = ['UNI', 'VIRCHOW2', 'GigaPath', 'H-Optimus-0']

for pair in ['MHIST->PCam', 'PCam->MHIST', 'CRC->BRACS', 'BRACS->CRC']:
    print(f'\n{"="*72}\n{pair}   raw ECE by source label fraction\n{"="*72}')
    for probe, df in [('linear', lin), ('mlp', mlp)]:
        t = (df[df.pair == pair]
             .groupby(['model','fraction'])['ece_fixed'].mean().unstack())
        if t.empty:
            continue
        print(f'\n  --- {probe} ---')
        print(t.round(3).to_string())
        mono = [m for m in t.index if all(np.diff(t.loc[m].values) >= -1e-9)]
        rising = [m for m in t.index if t.loc[m].iloc[-1] > t.loc[m].iloc[0]]
        print(f'  monoton artan : {sorted(mono)}')
        print(f'  net yukselen  : {sorted(rising)}')
        if pair == 'MHIST->PCam':
            hit = [m for m in CLAIM if m in rising]
            print(f'  >>> iddia edilen 4 modelden yukselen: {len(hit)}/4  {hit}')

print(f'\n\n{"="*72}\nMHIST->PCam, iddianin tam hali\n{"="*72}')
for m in CLAIM:
    line = f'  {m:>12}'
    for probe, df in [('linear', lin), ('mlp', mlp)]:
        v = (df[(df.pair == 'MHIST->PCam') & (df.model == m)]
             .groupby('fraction')['ece_fixed'].mean().sort_index())
        if len(v) >= 2:
            line += f'  |  {probe}: {v.iloc[0]:.3f} -> {v.iloc[-1]:.3f}'
    print(line)


MHIST->PCam   raw ECE by source label fraction

  --- linear ---
fraction      0.01   0.05   0.10   0.25   0.50   1.00
model                                                
CLIP         0.234  0.235  0.251  0.261  0.293  0.312
CONCH        0.229  0.270  0.321  0.398  0.439  0.455
GigaPath     0.244  0.300  0.345  0.385  0.437  0.462
H-Optimus-0  0.232  0.285  0.350  0.422  0.465  0.488
MIDNIGHT     0.218  0.212  0.256  0.251  0.268  0.304
PLIP         0.216  0.193  0.188  0.166  0.200  0.207
UNI          0.225  0.238  0.287  0.356  0.395  0.439
VIRCHOW2     0.231  0.306  0.370  0.409  0.435  0.453
  monoton artan : ['CLIP', 'CONCH', 'GigaPath', 'H-Optimus-0', 'UNI', 'VIRCHOW2']
  net yukselen  : ['CLIP', 'CONCH', 'GigaPath', 'H-Optimus-0', 'MIDNIGHT', 'UNI', 'VIRCHOW2']
  >>> iddia edilen 4 modelden yukselen: 4/4  ['UNI', 'VIRCHOW2', 'GigaPath', 'H-Optimus-0']

  --- mlp ---
fraction      0.01   0.05   0.10   0.25   0.50   1.00
model                                                
CLI

## 4. AUROC, degeneracy and post-hoc scaling under the MLP

In [5]:
print('=== OOD AUROC at 100% source labels ===')
for pair in ['MHIST->PCam', 'PCam->MHIST', 'CRC->BRACS', 'BRACS->CRC']:
    a = (lin[(lin.pair == pair) & (lin.fraction == 1.0)]
         .groupby('model')['auroc'].mean())
    b = (mlp[(mlp.pair == pair) & (mlp.fraction == 1.0)]
         .groupby('model')['auroc'].mean())
    j = pd.concat([a.rename('linear'), b.rename('mlp')], axis=1)
    j['delta'] = j['mlp'] - j['linear']
    print(f'\n{pair}:')
    print(j.round(4).to_string())

print('\n\n=== Degeneracy: share of cells collapsing to one class ===')
for probe, df in [('linear', lin), ('mlp', mlp)]:
    d = df.groupby('pair')['degenerate'].mean()
    print(f'  {probe:>7}: ' + '  '.join(f'{k} {v:.2f}' for k, v in d.items()))

print('\n\n=== Does source-fitted temperature help or hurt? ===')
for probe, df in [('linear', lin), ('mlp', mlp)]:
    col = 'ece_scaled_fixed'
    if col not in df.columns:
        continue
    sub = df[df[col].notna()]
    worse = (sub[col] > sub['ece_fixed']).mean()
    print(f'  {probe:>7}: scaling worsens ECE in {worse:.0%} of cells')

both = pd.concat([
    lin[['probe','model','pair','fraction','seed','auroc','ece_fixed',
         'ece_scaled_fixed','degeneracy_share']],
    mlp[['probe','model','pair','fraction','seed','auroc','ece_fixed',
         'ece_scaled_fixed','degeneracy_share']],
], ignore_index=True)
both.to_csv(f'{OUT_DIR}/ood_mlp_vs_linear.csv', index=False)
print('\nSaved -> ood_mlp.csv, ood_mlp_vs_linear.csv')

=== OOD AUROC at 100% source labels ===

MHIST->PCam:
             linear     mlp   delta
model                              
CLIP         0.6281  0.5759 -0.0522
CONCH        0.6955  0.7020  0.0064
GigaPath     0.4865  0.5431  0.0566
H-Optimus-0  0.5612  0.6454  0.0842
MIDNIGHT     0.4305  0.5857  0.1553
PLIP         0.5750  0.6132  0.0382
UNI          0.5894  0.6993  0.1100
VIRCHOW2     0.6848  0.7824  0.0976

PCam->MHIST:
             linear     mlp   delta
model                              
CLIP         0.4839  0.5779  0.0940
CONCH        0.6333  0.4845 -0.1487
GigaPath     0.5884  0.5784 -0.0101
H-Optimus-0  0.6448  0.6324 -0.0124
MIDNIGHT     0.4445  0.4293 -0.0153
PLIP         0.5479  0.5712  0.0233
UNI          0.4419  0.4355 -0.0065
VIRCHOW2     0.5845  0.6025  0.0180

CRC->BRACS:
             linear     mlp   delta
model                              
CLIP         0.6608  0.6226 -0.0382
CONCH        0.4285  0.4256 -0.0029
GigaPath     0.6024  0.5716 -0.0308
H-Optimus-0  0.5742

## 5. Reading the result

**If the trend survives for all four models.** The headline finding is a
property of the representations and the transfer setting, not of the probe.
This is the strongest outcome and should be stated with the MLP numbers
alongside the linear ones.

**If it survives for some.** Narrow the claim to those models and report the
Kendall agreement. A finding that holds for three of four is still a finding.

**If it does not survive.** Bound the claim to the linear-probe protocol in
the abstract and in Section 4, and report the MLP numbers. Combined with the
in-distribution result, the paper then makes a different and broader point:
calibration measurements are probe-sensitive while discriminative
measurements are not, which is itself worth knowing and is supported by three
independent measurements.